# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [3]:
import numpy as np
import pandas as pd

transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')

policy = 'do-nothing'
states = rewards.loc[rewards['action'] == policy, 'state'].tolist()
state_discomfort = (
    rewards.loc[rewards['action'] == policy]
    .set_index('state')['reward']
    .mul(-1)
    .to_dict()
)
policy_transitions = transitions.loc[transitions['action'] == policy]

# Value iteration for the fixed do-nothing policy, using discomfort as cost.
discomfort_values = {state: 0.0 for state in states}
for iteration in range(100000):
    updated_values = {}
    for state in states:
        next_rows = policy_transitions.loc[policy_transitions['state'] == state]
        expected_future_discomfort = sum(
            row['probability'] * discomfort_values[row['next_state']]
            for _, row in next_rows.iterrows()
        )
        updated_values[state] = state_discomfort[state] + expected_future_discomfort

    max_change = max(
        abs(updated_values[state] - discomfort_values[state])
        for state in states
    )
    discomfort_values = updated_values
    if max_change < 1e-12:
        break
else:
    raise RuntimeError('Value iteration did not converge.')

do_nothing_results = pd.DataFrame({
    'state': states,
    'expected_discomfort': [discomfort_values[state] for state in states]
})
do_nothing_results

,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,symptoms-1,6.666667
4,symptoms-2,5.000000
5,symptoms-3,1.666667
6,recovered,0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [4]:
do_nothing_results.to_csv(
    'do-nothing-discomfort.tsv',
    sep='\t',
    index=False
)
do_nothing_results

,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,symptoms-1,6.666667
4,symptoms-2,5.000000
5,symptoms-3,1.666667
6,recovered,0.000000


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [5]:
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')

states = rewards['state'].drop_duplicates().tolist()
actions = rewards['action'].drop_duplicates().tolist()

transition_totals = transitions.groupby(['action', 'state'])['probability'].sum()
assert np.allclose(transition_totals.to_numpy(), 1.0)

reward_lookup = rewards.set_index(['action', 'state'])['reward']
value_function = {state: 0.0 for state in states}

# Bellman optimality iteration; rewards are negative discomfort, so maximize them.
for iteration in range(100000):
    updated_values = {}
    for state in states:
        action_values = []
        for action in actions:
            next_rows = transitions.loc[
                (transitions['action'] == action) &
                (transitions['state'] == state)
            ]
            expected_return = reward_lookup[(action, state)] + sum(
                row['probability'] * value_function[row['next_state']]
                for _, row in next_rows.iterrows()
            )
            action_values.append(expected_return)
        updated_values[state] = max(action_values)

    max_change = max(
        abs(updated_values[state] - value_function[state])
        for state in states
    )
    value_function = updated_values
    if max_change < 1e-12:
        break
else:
    raise RuntimeError('Optimal value iteration did not converge.')

optimal_actions = {}
for state in states:
    action_values = {}
    for action in actions:
        next_rows = transitions.loc[
            (transitions['action'] == action) &
            (transitions['state'] == state)
        ]
        action_values[action] = reward_lookup[(action, state)] + sum(
            row['probability'] * value_function[row['next_state']]
            for _, row in next_rows.iterrows()
        )
    optimal_actions[state] = max(action_values, key=action_values.get)

optimal_actions

{'exposed-1': 'sleep-8',
 'exposed-2': 'sleep-8',
 'exposed-3': 'sleep-8',
 'symptoms-1': 'drink-oj',
 'symptoms-2': 'drink-oj',
 'symptoms-3': 'drink-oj',
 'recovered': 'do-nothing'}

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [6]:
minimum_discomfort_actions = pd.DataFrame({
    'state': states,
    'action': [optimal_actions[state] for state in states]
})
minimum_discomfort_actions.to_csv(
    'minimum-discomfort-actions.tsv',
    sep='\t',
    index=False
)
minimum_discomfort_actions

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [7]:
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')
minimum_discomfort_actions = pd.read_csv(
    'minimum-discomfort-actions.tsv', sep='\t'
)

states = rewards['state'].drop_duplicates().tolist()
policy = dict(zip(
    minimum_discomfort_actions['state'],
    minimum_discomfort_actions['action']
))
assert set(policy) == set(states)

reward_lookup = rewards.set_index(['action', 'state'])['reward']
policy_transitions = transitions[
    transitions.apply(
        lambda row: policy[row['state']] == row['action'], axis=1
    )
]
transition_totals = policy_transitions.groupby('state')['probability'].sum()
assert np.allclose(transition_totals.reindex(states).to_numpy(), 1.0)

# Evaluate the fixed optimal policy in discomfort units.
expected_discomfort_by_state = {state: 0.0 for state in states}
for iteration in range(100000):
    updated_discomfort = {}
    for state in states:
        action = policy[state]
        state_transitions = policy_transitions[
            policy_transitions['state'] == state
        ]
        expected_future = sum(
            row['probability'] * expected_discomfort_by_state[row['next_state']]
            for _, row in state_transitions.iterrows()
        )
        updated_discomfort[state] = (
            -reward_lookup[(action, state)] + expected_future
        )

    max_change = max(
        abs(updated_discomfort[state] - expected_discomfort_by_state[state])
        for state in states
    )
    expected_discomfort_by_state = updated_discomfort
    if max_change < 1e-12:
        break
else:
    raise RuntimeError('Policy evaluation did not converge.')

minimum_discomfort_values = pd.DataFrame({
    'state': states,
    'expected_discomfort': [
        expected_discomfort_by_state[state] for state in states
    ]
})
minimum_discomfort_values

,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,symptoms-1,6.00
4,symptoms-2,4.50
5,symptoms-3,1.50
6,recovered,0.00


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [8]:
minimum_discomfort_values.to_csv(
    'minimum-discomfort-values.tsv',
    sep='\t',
    index=False
)
minimum_discomfort_values

,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,symptoms-1,6.00
4,symptoms-2,4.50
5,symptoms-3,1.50
6,recovered,0.00


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [9]:
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')

duration_rewards = rewards[['action', 'state']].copy()
duration_rewards['reward'] = np.where(
    duration_rewards['state'].str.startswith('symptoms-'),
    -1.0,
    0.0
)
duration_rewards

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-1.0
6,do-nothing,recovered,0.0
7,drink-oj,exposed-1,0.0
8,drink-oj,exposed-2,0.0
9,drink-oj,exposed-3,0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [10]:
duration_rewards.to_csv(
    'duration-rewards.tsv',
    sep='\t',
    index=False
)
duration_rewards.head()

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [11]:
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
duration_rewards = pd.read_csv('duration-rewards.tsv', sep='\t')

states = duration_rewards['state'].drop_duplicates().tolist()
actions = duration_rewards['action'].drop_duplicates().tolist()
reward_lookup = duration_rewards.set_index(['action', 'state'])['reward']
value_function = {state: 0.0 for state in states}

# Maximize the duration rewards, which are -1 per sick day and 0 otherwise.
for iteration in range(100000):
    updated_values = {}
    for state in states:
        action_values = {}
        for action in actions:
            next_rows = transitions.loc[
                (transitions['action'] == action) &
                (transitions['state'] == state)
            ]
            action_values[action] = reward_lookup[(action, state)] + sum(
                row['probability'] * value_function[row['next_state']]
                for _, row in next_rows.iterrows()
            )
        updated_values[state] = max(action_values.values())

    max_change = max(
        abs(updated_values[state] - value_function[state])
        for state in states
    )
    value_function = updated_values
    if max_change < 1e-12:
        break
else:
    raise RuntimeError('Minimum-duration value iteration did not converge.')

minimum_duration_actions = {}
for state in states:
    action_values = {}
    for action in actions:
        next_rows = transitions.loc[
            (transitions['action'] == action) &
            (transitions['state'] == state)
        ]
        action_values[action] = reward_lookup[(action, state)] + sum(
            row['probability'] * value_function[row['next_state']]
            for _, row in next_rows.iterrows()
        )
    minimum_duration_actions[state] = max(
        action_values, key=action_values.get
    )

minimum_duration_actions

{'exposed-1': 'sleep-8',
 'exposed-2': 'sleep-8',
 'exposed-3': 'sleep-8',
 'symptoms-1': 'do-nothing',
 'symptoms-2': 'do-nothing',
 'symptoms-3': 'do-nothing',
 'recovered': 'do-nothing'}

Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [12]:
minimum_duration_actions_df = pd.DataFrame({
    'state': states,
    'action': [minimum_duration_actions[state] for state in states]
})
minimum_duration_actions_df.to_csv(
    'minimum-duration-actions.tsv',
    sep='\t',
    index=False
)
minimum_duration_actions_df

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [13]:
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
duration_rewards = pd.read_csv('duration-rewards.tsv', sep='\t')
minimum_duration_actions_df = pd.read_csv(
    'minimum-duration-actions.tsv', sep='\t'
)

states = duration_rewards['state'].drop_duplicates().tolist()
policy = dict(zip(
    minimum_duration_actions_df['state'],
    minimum_duration_actions_df['action']
))
assert set(policy) == set(states)

reward_lookup = duration_rewards.set_index(['action', 'state'])['reward']
policy_transitions = transitions[
    transitions.apply(
        lambda row: policy[row['state']] == row['action'], axis=1
    )
]
transition_totals = policy_transitions.groupby('state')['probability'].sum()
assert np.allclose(transition_totals.reindex(states).to_numpy(), 1.0)

# Evaluate the fixed minimum-duration policy in sick-day units.
expected_sick_days_by_state = {state: 0.0 for state in states}
for iteration in range(100000):
    updated_sick_days = {}
    for state in states:
        action = policy[state]
        state_transitions = policy_transitions[
            policy_transitions['state'] == state
        ]
        expected_future = sum(
            row['probability'] * expected_sick_days_by_state[row['next_state']]
            for _, row in state_transitions.iterrows()
        )
        updated_sick_days[state] = (
            -reward_lookup[(action, state)] + expected_future
        )

    max_change = max(
        abs(updated_sick_days[state] - expected_sick_days_by_state[state])
        for state in states
    )
    expected_sick_days_by_state = updated_sick_days
    if max_change < 1e-12:
        break
else:
    raise RuntimeError('Expected sick days did not converge.')

minimum_duration_days = pd.DataFrame({
    'state': states,
    'expected_sick_days': [
        expected_sick_days_by_state[state] for state in states
    ]
})
minimum_duration_days

,state,expected_sick_days
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,symptoms-1,10.000000
4,symptoms-2,6.666667
5,symptoms-3,3.333333
6,recovered,0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [14]:
minimum_duration_days.to_csv(
    'minimum-duration-days.tsv',
    sep='\t',
    index=False
)
minimum_duration_days

,state,expected_sick_days
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,symptoms-1,10.000000
4,symptoms-2,6.666667
5,symptoms-3,3.333333
6,recovered,0.000000


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [15]:
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')
minimum_duration_actions_df = pd.read_csv(
    'minimum-duration-actions.tsv', sep='\t'
)
minimum_discomfort_actions = pd.read_csv(
    'minimum-discomfort-actions.tsv', sep='\t'
)

states = rewards['state'].drop_duplicates().tolist()
reward_lookup = rewards.set_index(['action', 'state'])['reward']

def evaluate_discomfort(policy_actions):
    policy = dict(zip(policy_actions['state'], policy_actions['action']))
    assert set(policy) == set(states)
    policy_transitions = transitions[
        transitions.apply(
            lambda row: policy[row['state']] == row['action'], axis=1
        )
    ]
    transition_totals = policy_transitions.groupby('state')['probability'].sum()
    assert np.allclose(transition_totals.reindex(states).to_numpy(), 1.0)

    values = {state: 0.0 for state in states}
    for iteration in range(100000):
        updated_values = {}
        for state in states:
            action = policy[state]
            state_transitions = policy_transitions[
                policy_transitions['state'] == state
            ]
            expected_future = sum(
                row['probability'] * values[row['next_state']]
                for _, row in state_transitions.iterrows()
            )
            updated_values[state] = (
                -reward_lookup[(action, state)] + expected_future
            )

        max_change = max(
            abs(updated_values[state] - values[state]) for state in states
        )
        values = updated_values
        if max_change < 1e-12:
            return values

    raise RuntimeError('Policy discomfort evaluation did not converge.')

speed_discomfort = evaluate_discomfort(minimum_duration_actions_df)
minimize_discomfort = evaluate_discomfort(minimum_discomfort_actions)

policy_comparison = pd.DataFrame({
    'state': states,
    'speed_discomfort': [speed_discomfort[state] for state in states],
    'minimize_discomfort': [minimize_discomfort[state] for state in states]
})
policy_comparison

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,symptoms-1,6.666667,6.00
4,symptoms-2,5.000000,4.50
5,symptoms-3,1.666667,1.50
6,recovered,0.000000,0.00


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [16]:
policy_comparison.to_csv(
    'policy-comparison.tsv',
    sep='\t',
    index=False
)
policy_comparison

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,symptoms-1,6.666667,6.00
4,symptoms-2,5.000000,4.50
5,symptoms-3,1.666667,1.50
6,recovered,0.000000,0.00


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used NumPy and pandas for loading the supplied TSV files, performing value iteration, and creating the required output tables. GitHub Copilot Chat in Visual Studio Code assisted with interpreting the instructions, implementing and checking the MDP calculations, and validating the output files.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)